# Deep Analysis: LLM IaC Generation Errors & DevOps Simulation
This notebook performs a multi-model evaluation of Infrastructure-as-Code (IaC) generation. It links scenario outcomes (`results.csv`) with the granular step-by-step failures (`error_history.csv`) to extract insights into *why* LLMs fail and *what they lack*.

**Key Analytical Goals:**
1. Cross-Model Performance (Pass@1 vs. Iterative Success)
2. Pipeline Stage Bottlenecks (Syntax vs. Live Deployment)
3. Model Stubbornness (Failing to resolve the exact same error across iterations)
4. Resource Vulnerability (Which cloud resources are hardest to generate?)
5. Identification of Research Gaps

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import ast
import re
from collections import Counter

# Set plotting style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context("notebook", font_scale=1.1)

# 1. Load Results Data
result_files = [
    './result/iterative_claude-3-5-sonnet-20241022_results.csv',
    './result/iterative_claude-3-7-sonnet-20250219_results.csv',
    './result/iterative_o3-mini_results.csv',
    './result/iterative_gpt-4o_results.csv',
    './result/iterative_deepseek-reasoner_results.csv',
    './result/iterative_deepseek-chat_results.csv'
]

results_df = pd.DataFrame()
for file in result_files:
    temp_df = pd.read_csv(file)
    # Extract model name from filename for tracking
    model_name = file.replace('./result/iterative_', '').replace('_results.csv', '')
    temp_df['model'] = model_name
    results_df = pd.concat([results_df, temp_df], ignore_index=True)

# Clean up success column boolean
results_df['success'] = results_df['success'].astype(str).str.upper() == 'TRUE'

# 2. Load Error History Data
history_files = [
    './error_tracking/claude-3-5-sonnet-20241022_error_history.csv',
    './error_tracking/claude-3-7-sonnet-20250219_error_history.csv',
    './error_tracking/o3-mini_error_history.csv',
    './error_tracking/gpt-4o_error_history.csv',
    './error_tracking/deepseek-reasoner_error_history.csv',
    './error_tracking/deepseek-chat_error_history.csv'
]

history_df = pd.DataFrame()
for file in history_files:
    temp_df = pd.read_csv(file)
    history_df = pd.concat([history_df, temp_df], ignore_index=True)

# 3. Data Cleaning
history_df['row_number'] = pd.to_numeric(history_df['row_number'], errors='coerce')
results_df['row_number'] = pd.to_numeric(results_df['row_number'], errors='coerce')

# Map the generic 'llm_model' from history to match the 'model' key in results
history_df['model_key'] = history_df['llm_model'].apply(
    lambda x: 'claude-3-5' if 'claude-3-5' in str(x).lower() else
    lambda x: 'claude-3-7' if 'claude-3-7' in str(x).lower() else
              'o3-mini' if 'o3-mini' in str(x).lower() else
              'gpt-4o' if 'gpt' in str(x).lower() else
              'deepseek-reasoner' if 'deepseek-reasoner' in str(x).lower() else
              'deepseek-chat' if 'deepseek-chat' in str(x).lower() else str(x)
)

print(f"Loaded {len(results_df)} total scenario results across {results_df['model'].nunique()} models.")
print(f"Loaded {len(history_df)} granular error events.")

## 1. Overall Pass Rates and Pass@k
If an LLM requires 12 iterations to deploy a database, is it truly autonomous? Here we compare the zero-shot (Pass@1) capabilities vs the final success rate after exhausting all feedback attempts.

In [ ]:
plt.figure(figsize=(14, 6))

# Calculate Pass@1 and Overall Success
metrics = []
for model in results_df['model'].unique():
    model_data = results_df[results_df['model'] == model]
    total = len(model_data)
    if total == 0: continue
        
    overall_pass = model_data['success'].sum()
    pass_at_1 = len(model_data[(model_data['success'] == True) & (model_data['total_iterations'] == 1)])
    
    metrics.append({
        'Model': model,
        'Pass@1 (%)': (pass_at_1 / total) * 100,
        'Overall Pass (%)': (overall_pass / total) * 100
    })

metrics_df = pd.DataFrame(metrics).melt(id_vars='Model', var_name='Metric', value_name='Success Rate (%)')

# Plot
sns.barplot(data=metrics_df, x='Model', y='Success Rate (%)', hue='Metric', palette='Blues_d')
plt.title('Zero-Shot vs. Iterative Deployment Success by Model')
plt.ylim(0, 100)
for p in plt.gca().patches:
    plt.gca().annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()), 
                       ha='center', va='bottom', fontsize=10, xytext=(0, 5), textcoords='offset points')
plt.show()

## 2. Where Do Models Fail? (DevOps Pipeline Stages)
Do models fail because they write invalid YAML/JSON (Syntax), or do they write syntactically valid code that hallucinates non-existent cloud resources (Deployment)?

In [ ]:
# Filter only failed scenarios from results
failed_results = results_df[results_df['success'] == False].copy()

failed_results['failed_at_stage'] = failed_results['failed_at_stage'].fillna('Unknown').astype(str)
failed_results['model'] = failed_results['model'].fillna('Unknown').astype(str)

plt.figure(figsize=(12, 6))
sns.countplot(data=failed_results, x='model', hue='failed_at_stage', palette='rocket')
plt.title('Terminal Failure Stage by Model')
plt.xlabel('Model')
plt.ylabel('Number of Terminal Failures')
plt.legend(title='Failed At Stage')
plt.tight_layout()
plt.show()

history_df_clean = history_df.copy()
history_df_clean['error_stage'] = history_df_clean['error_stage'].fillna('Unknown').astype(str)
history_df_clean['model_key'] = history_df_clean['model_key'].fillna('Unknown').astype(str)

# Look at granular error distribution from history_df
plt.figure(figsize=(12, 6))
sns.countplot(data=history_df_clean, y='error_stage', hue='model_key', palette='Set2')
plt.title('Total Volume of Errors Generated Across All Iterations (By Stage)')
plt.xlabel('Total Error Events Logged')
plt.ylabel('Pipeline Stage')
plt.tight_layout()
plt.show()

## 3. Model Stubbornness (Repeating Errors)
A critical limitation in current LLM agents is getting stuck in local optima. If the exact same error message on the exact same resource happens in iteration `N` and `N+1`, the LLM has completely failed to integrate the feedback.

In [ ]:
# 1. Clean the dataframe to ensure no missing grouping keys
history_sorted = history_df.dropna(subset=['error_message', 'model_key', 'row_number']).copy()

# 2. Force strict types (this prevents Numpy/Pandas from hanging during sorting and grouping)
history_sorted['row_number'] = history_sorted['row_number'].astype(int)
history_sorted['iteration_number'] = pd.to_numeric(history_sorted['iteration_number'], errors='coerce').fillna(0).astype(int)
history_sorted['resource_name'] = history_sorted['resource_name'].fillna('Unknown_Resource').astype(str)
history_sorted['error_message'] = history_sorted['error_message'].astype(str)

# 3. Sort chronologically per scenario
history_sorted = history_sorted.sort_values(by=['model_key', 'row_number', 'iteration_number'])

# 4. Shift to find exact repeated errors on the exact same resource
history_sorted['prev_error'] = history_sorted.groupby(['model_key', 'row_number', 'resource_name'])['error_message'].shift(1)

# 5. Flag exact repeats (ignoring boilerplate nulls)
ignore_list = ['N/A', 'Unknown', 'nan', 'None', '']
history_sorted['is_repeat'] = (history_sorted['error_message'] == history_sorted['prev_error']) & (~history_sorted['error_message'].isin(ignore_list))

# 6. Calculate model stubborness rate
repeat_rates = history_sorted.groupby('model_key')['is_repeat'].mean() * 100

# Plotting
plt.figure(figsize=(10, 5))
sns.barplot(x=repeat_rates.index, y=repeat_rates.values, palette='magma')
plt.title('Model Stubbornness Rate (% of errors that are exact repeats of previous iteration)')
plt.ylabel('Repeat Error Rate (%)')
for p in plt.gca().patches:
    plt.gca().annotate(f"{p.get_height():.1f}%", (p.get_x() + p.get_width() / 2., p.get_height()), 
                       ha='center', va='bottom')
plt.tight_layout()
plt.show()


print("Example of a repeated error loop:")
# Display a sample of the repeated loops
repeats_df = history_sorted[history_sorted['is_repeat'] == True]
if not repeats_df.empty:
    pd.set_option('display.max_colwidth', None)
    display(repeats_df[['model_key', 'row_number', 'iteration_number', 'resource_name', 'error_message']].head())
else:
    print("No exact repeated errors found!")

## 4. Resource Vulnerability & Keyword Analysis
Which AWS resources cause the deployment to crash? We extract the text from the API rejection messages to see *why* deployment fails (e.g., hallucinated Engine versions, missing Docker dependencies).

In [ ]:
# Most problematic resources
deployment_errors = history_df[history_df['error_stage'] == 'deployment']
top_resources = deployment_errors['resource_name'].value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_resources.values, y=top_resources.index, palette='viridis')
plt.title('Top 10 Most Problematic Cloud Resources (Deployment Phase)')
plt.xlabel('Total Deployment Rejections')
plt.ylabel('Logical Resource Name')
plt.show()

# Extract Hallucination Evidence
print("Evidence of Hallucinations / Logic Errors in Deployment:")
hallucinations = deployment_errors[deployment_errors['error_message'].str.contains('Cannot find version|does not exist|InvalidRequest', na=False, case=False)]
pd.set_option('display.max_colwidth', None)
display(hallucinations[['model_key', 'resource_name', 'error_message']].drop_duplicates().head(5))

## 5. Extracted Insights & Gaps in Current LLM IaC Research
Based on the data linked across `results` and `error_history`, here are the core gaps in current LLM capabilities and academic research for IaC:

### 1. The "Whack-A-Mole" Problem vs. Holistic Reasoning
**Observation:** Models often fix one syntax error (e.g., adding `DeletionPolicy`) only to break dependencies (e.g., IAM circular references) in the next iteration. The high "stubbornness" rate shows models fail to reason globally about the infrastructure state. 
* **Research Gap:** Current iterative evaluation limits focus on sequential string-based error feedback. There is a lack of research into injecting **Graph-based state tracking** into the prompt context so the LLM understands how a single change cascades across network boundaries and IAM roles.

### 2. Hallucination of Transient Cloud State (Engine Versions & AMIs)
**Observation:** In the `error_history`, deployment fails heavily on things like *“Cannot find version 5.7.mysql_aurora.2.09.2”* (from the GPT-4o dataset). 
* **Research Gap:** LLMs rely on their parametric memory, which freezes at a point in time. Cloud provider APIs constantly deprecate old RDS engine versions or AMI IDs. Current IaC agents lack **active, pre-deployment API polling**. Research needs to bridge LLMs with tools that can securely query the AWS/Azure CLI *before* writing the template to fetch valid, up-to-date enums.

### 3. Execution Environment Dependencies
**Observation:** Errors like *"Docker not available"* appear when creating Lambda functions using local asset packaging. 
* **Research Gap:** LLMs write code assuming standard developer workstations. DevOps simulation benchmarks rarely account for the local state of the CI/CD runner executing the IaC (e.g., missing Docker daemons for Lambda builds, missing pre-compiled binaries). Future research must explore "Environment-Aware IaC Generation," where the LLM writes fallback logic or probes the runner environment.

### 4. Over-reliance on "Live Deployment" for Validation
**Observation:** As seen in the graphs, terminal failures overwhelmingly occur in the `deployment` stage. It takes tens of minutes to deploy and rollback a failed RDS or VPC setup.
* **Research Gap:** Relying on live cloud deployment for the feedback loop is slow, costly, and dangerous. There is a massive gap in research utilizing **Local Cloud Emulators (like LocalStack)** combined with static policy checkers (like Checkov/OPA) as the primary feedback loop for LLMs, effectively shifting the IaC evaluation left.